# 5 autogen: practice

This module/project pack is extracted from the local files listed below. Functions and line numbers are source locations; the original program is not executed by practice.

Use this with [the walkthrough](walkthrough.md).

**Source:**

- [5_autogen/agent.py](../../../../../5_autogen/agent.py)
- [5_autogen/creator.py](../../../../../5_autogen/creator.py)
- [5_autogen/messages.py](../../../../../5_autogen/messages.py)
- [5_autogen/world.py](../../../../../5_autogen/world.py)

**Run one cell at a time:** predict → run with Shift+Enter → explain → change one thing. Select the root project .venv kernel.

**Offline adaptation:** code below uses fictional data or local libraries. It never executes the linked source, reads a credential file, sends messages, starts a UI/server, or changes the course databases. Simulations are labeled; they do not verify external integrations.

## 1. Configuration stays outside executable examples

Environment-variable names identify settings. Values belong in .environment/.env and are loaded deliberately. This exercise inspects only the path and simulated presence flags; it reads no private file or variable values.

**Predict:** Does finding a file prove that authentication works?

In [ ]:
from pathlib import Path
# Find the repository from a kernel started inside it.
root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "1_foundations" / "3_lab3.ipynb").is_file()), None)
if root is None:
    raise FileNotFoundError("Start your kernel inside ai_agents")
configuration_path = root / ".environment" / ".env"
print("Configuration path:", configuration_path)
simulated_presence = {"OPENAI_API_KEY": True, "GOOGLE_API_KEY": False}
print(simulated_presence)  # Booleans only; these are fictional flags.
assert configuration_path.name == ".env"

**Check:** No. File existence, variable presence, and successful provider authentication are separate checks.

**Change one thing:** Explain the difference between a missing file, a missing variable, and a provider rejecting authentication. Do not print token prefixes or values.

## 2. Messages: instructions, earlier turns, and the new question

The model request is a list of messages. A role says who is speaking; content holds the text. The system instruction and prior conversation serve different purposes.

**Predict:** How many messages are sent on the follow-up turn?

In [ ]:
instructions = "Explain Python with small examples."
history = [
    {"role": "user", "content": "What is a list?"},
    {"role": "assistant", "content": "An ordered collection of values."},
]
# List addition preserves the order: instruction, old turns, new question.
messages = [{"role": "system", "content": instructions}] + history + [
    {"role": "user", "content": "How is a dictionary different?"}
]
print([item["role"] for item in messages])
assert len(messages) == 4

**Check:** Four: system, user, assistant, user. With empty history, only two remain.

**Change one thing:** Set history to an empty list. Explain what is lost while the system instruction remains.

## 3. Client, request, and response are separate objects

Constructing a client prepares a connection interface. A request invokes it. The returned response contains choices, a message, and its content. This fixture imitates that nested shape.

**Predict:** What does choices[0] select?

In [ ]:
from types import SimpleNamespace
response = SimpleNamespace(choices=[
    SimpleNamespace(message=SimpleNamespace(content="The answer is 4."))
])
answer = response.choices[0].message.content
print(answer)
assert answer == "The answer is 4."

**Check:** The first returned choice. This is simulated response data, not a model call.

**Change one thing:** Add a second choice with different text. Read choices[1] and explain why choosing an index matters.

## 4. Await vs. concurrent work

Calling an async function creates a coroutine. Await executes/waits for it. gather schedules independent coroutines and returns results in input order. Notebook cells support top-level await.

**Predict:** Will results follow completion order or input order?

In [ ]:
import asyncio
async def work(label, delay):
    await asyncio.sleep(delay)  # Yield control while waiting.
    return label
results = await asyncio.gather(work("A", 0.002), work("B", 0))
print(results)
assert results == ["A", "B"]

**Check:** Input order. In a standalone script, wrap work in async def main and call asyncio.run(main()); do not do that inside a running notebook loop.

**Change one thing:** Run the two functions with separate awaits. Explain why a dependent stage cannot start until it has its input.

## 5. Typed messages and agent identities

A runtime delivers a message to an identified handler. The handler owns its work. This dictionary-based simulation exposes type/key routing without starting AutoGen or network listeners.

**Predict:** Will an unregistered type resolve?

In [ ]:
from dataclasses import dataclass
@dataclass
class Message:
    content: str
registry = {("simple_agent", "default"): lambda message: Message("Received: " + message.content)}
recipient = ("simple_agent", "default")  # Agent type and instance key.
response = registry[recipient](Message("Hello"))
print(response.content)
assert response.content == "Received: Hello"

**Check:** No. Registration must precede delivery. The simulation does not verify an AutoGen runtime.

**Change one thing:** Register a second handler and send the response to it. Explain which code routes and which code does the work.

## 6. Host and workers have different responsibilities

A distributed host connects workers; workers register agents and handle requests. A stop sequence must clean up both. This logical model opens no port and starts no process.

**Predict:** If both workers register different agents, who handles the judge request?

In [ ]:
workers = {"worker_a": {"researcher"}, "worker_b": {"judge"}}
def locate(agent_type):
    matches = [worker for worker, agents in workers.items() if agent_type in agents]
    if len(matches) != 1:
        raise LookupError("Need exactly one matching worker in this fixture")
    return matches[0]
print(locate("judge"))
cleanup_order = ["worker_a stopped", "worker_b stopped", "host stopped"]
print(cleanup_order)
assert locate("judge") == "worker_b"

**Check:** worker_b. The real source uses gRPC; this exercise only models the delivery/lifetime concept.

**Change one thing:** Add another agent type to worker_a and locate it. Explain how a host connection failure differs from missing registration.

## 7. Generated code is an output before it is execution

A coding workflow designs, writes, and reviews code. Generated files or Docker execution are side effects. First inspect and validate a harmless code string locally.

**Predict:** Does parsing Python text execute it?

In [ ]:
import ast
proposed = "def add(a, b):\n    return a + b\n"
tree = ast.parse(proposed)  # Parse syntax only; do not execute generated text.
functions = [node.name for node in tree.body if isinstance(node, ast.FunctionDef)]
print(functions)
assert functions == ["add"]
checks = {"has_function": True, "has_test_plan": False}
print("Review:", checks)

**Check:** No. Syntax validity does not prove behavior or make generated code safe to execute.

**Change one thing:** Write a test plan for the intended function. Explain which separate step would run code and where the source uses Docker.

## Rebuild from memory

1. Recreate the main data-flow step in a new scratch cell.
2. Add one case that should fail and identify its exception/stop condition.
3. Compare your version with the linked source cells or functions.
4. Explain where a real model or service would enter, and what it could change.

For deliberate original-source execution, see the shared [configuration guide](../../../configuration.md). All provider/transport integrations remain UNVERIFIED at runtime.